# 1. Préambule

## 1.1 Problématique

La très jeune start-up de l'AgriTech, nommée "**Fruits**!", <br />
cherche à proposer des solutions innovantes pour la récolte des fruits.

La volonté de l’entreprise est de préserver la biodiversité des fruits <br />
en permettant des traitements spécifiques pour chaque espèce de fruits <br />
en développant des robots cueilleurs intelligents.

La start-up souhaite dans un premier temps se faire connaître en mettant <br />
à disposition du grand public une application mobile qui permettrait aux <br />
utilisateurs de prendre en photo un fruit et d'obtenir des informations sur ce fruit.

Pour la start-up, cette application permettrait de sensibiliser le grand public <br /> 
à la biodiversité des fruits et de mettre en place une première version du moteur <br />
de classification des images de fruits.

De plus, le développement de l’application mobile permettra de construire <br />
une première version de l'architecture **Big Data** nécessaire.

## 1.2 Objectifs dans ce projet

1. Développer une première chaîne de traitement des données qui <br />
   comprendra le **preprocessing** et une étape de **réduction de dimension**.
2. Tenir compte du fait que <u>le volume de données va augmenter <br />
   très rapidement</u> après la livraison de ce projet, ce qui implique de:
 - Déployer le traitement des données dans un environnement **Big Data**
 - Développer les scripts en **pyspark** pour effectuer du **calcul distribué**

## 1.3 Déroulement des étapes du projet

Le projet va être réalisé en 2 temps, dans deux environnements différents. <br />
Nous allons dans un premier temps développer et exécuter notre code en local, <br />
en travaillant sur un nombre limité d'images à traiter.

Une fois les choix techniques validés, nous déploierons notre solution <br />
dans un environnement Big Data en mode distribué.

## 1.4 Transfert Learning

L'énoncé du projet nous demande également de <br />
réaliser une première chaîne de traitement <br />
des données qui comprendra le preprocessing et <br />
une étape de réduction de dimension.

Il est également précisé qu'il n'est pas nécessaire <br />
d'entraîner un modèle pour le moment.

Nous décidons de partir sur une solution de **transfert learning**.

Simplement, le **transfert learning** consiste <br />
à utiliser la connaissance déjà acquise <br />
par un modèle entraîné (ici **MobileNetV2**) pour <br />
l'adapter à notre problématique.

Nous allons fournir au modèle nos images, et nous allons <br />
<u>récupérer l'avant dernière couche</u> du modèle.<br />
En effet la dernière couche de modèle est une couche softmax <br />
qui permet la classification des images ce que nous ne <br />
souhaitons pas dans ce projet.

L'avant dernière couche correspond à un **vecteur <br />
réduit** de dimension (1,1,1280).

Cela permettra de réaliser une première version du moteur <br />
pour la classification des images des fruits.

**MobileNetV2** a été retenu pour sa <u>rapidité d'exécution</u>, <br />
particulièrement adaptée pour le traitement d'un gros volume <br />
de données ainsi que la <u>faible dimensionnalité du vecteur <br />
de caractéristique en sortie</u> (1,1,1280)

# Déploiement de la solution en local

# 2. Préparation des données et modèle


## 2.1 Environnement de travail

Pour des raisons de simplicité, nous développons dans un environnement <br />
Linux Unbuntu (exécuté depuis une machine Windows dans une machine virtuelle)
* Pour installer une machine virtuelle :  https://www.malekal.com/meilleurs-logiciels-de-machine-virtuelle-gratuits-ou-payants/

## 2.2 Installation de Spark

[La première étape consiste à installer Spark ](https://computingforgeeks.com/how-to-install-apache-spark-on-ubuntu-debian/)

## 2.3 Installation des packages

<u>On installe ensuite à l'aide de la commande **pip** <br />
les packages qui nous seront nécessaires</u> :

In [2]:
!pip install Pandas pillow tensorflow pyspark pyarrow

## 2.4 Import des librairies

In [3]:
import pandas as pd
from PIL import Image
import numpy as np
import io
import os

import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras import Model
from pyspark.sql.functions import col, pandas_udf, PandasUDFType, element_at, split, udf
from pyspark.sql import SparkSession
from pyspark.ml.feature import PCA
from pyspark.ml.linalg import Vectors, VectorUDT

I0000 00:00:1781263216.370253   98435 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781263217.360615   98435 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781263224.881871   98435 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## 2.5 Définition des PATH pour charger les images <br /> et enregistrer les résultats

Dans cette version locale, nous partons du principe que les données <br />
sont stockées dans le même répertoire que le notebook.<br />
Nous n'utilisons qu'un extrait de **300 images** à traiter dans cette <br />
première version en local.<br />
L'extrait des images à charger est stockée dans le dossier **Test1**.<br />
Nous enregistrerons le résultat de notre traitement <br />
dans le dossier "**Results**"

In [4]:
PATH = os.getcwd()
PATH_Data = PATH+'/data/Test1'
PATH_Result = PATH+'/data/Results'
print('PATH:        '+\
      PATH+'\nPATH_Data:   '+\
      PATH_Data+'\nPATH_Result: '+PATH_Result)

PATH:        /home/steeve/Projets OCR/P9_BigData/Notebook
PATH_Data:   /home/steeve/Projets OCR/P9_BigData/Notebook/data/Test1
PATH_Result: /home/steeve/Projets OCR/P9_BigData/Notebook/data/Results


## 2.6 Création de la SparkSession

L’application Spark est contrôlée grâce à un processus de pilotage (driver process) appelé **SparkSession**. <br />
<u>Une instance de **SparkSession** est la façon dont Spark exécute les fonctions définies par l’utilisateur <br />
dans l’ensemble du cluster</u>. <u>Une SparkSession correspond toujours à une application Spark</u>.

<u>Ici nous créons une session spark en spécifiant dans l'ordre</u> :
 1. un **nom pour l'application**, qui sera affichée dans l'interface utilisateur Web Spark "**P8**"
 2. que l'application doit s'exécuter **localement**. <br />
   Nous ne définissons pas le nombre de cœurs à utiliser (comme .master('local[4]) pour 4 cœurs à utiliser), <br />
   nous utiliserons donc tous les cœurs disponibles dans notre processeur.<br />
 3. une option de configuration supplémentaire permettant d'utiliser le **format "parquet"** <br />
   que nous utiliserons pour enregistrer et charger le résultat de notre travail.
 4. vouloir **obtenir une session spark** existante ou si aucune n'existe, en créer une nouvelle

In [5]:
spark = (SparkSession
             .builder
             .appName('P8')
             .master('local')
             .config("spark.sql.parquet.writeLegacyFormat", 'true')
             .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/12 13:20:32 WARN Utils: Your hostname, SteeveO, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/12 13:20:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/12 13:20:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<u>Nous créons également la variable "**sc**" qui est un **SparkContext** issue de la variable **spark**</u> :

In [6]:
sc = spark.sparkContext

<u>Affichage des informations de Spark en cours d'execution</u> :

In [7]:
spark

# 3 Traitement des données

<u>Dans la suite de notre flux de travail, <br />
nous allons successivement</u> :
1. Préparer nos données
    1. Importer les images dans un dataframe **pandas UDF**
    2. Associer aux images leur **label**
    3. Préprocesser en **redimensionnant nos images pour <br />
       qu'elles soient compatibles avec notre modèle**
2. Préparer notre modèle
    1. Importer le modèle **MobileNetV2**
    2. Créer un **nouveau modèle** dépourvu de la dernière couche de MobileNetV2
3. Définir le processus de chargement des images et l'application <br />
   de leur featurisation à travers l'utilisation de pandas UDF
3. Exécuter les actions d'extraction de features
4. Enregistrer le résultat de nos actions
5. Tester le bon fonctionnement en chargeant les données enregistrées




## 3.7. Chargement des données

Les images sont chargées au format binaire, ce qui offre, <br />
plus de souplesse dans la façon de prétraiter les images.

Avant de charger les images, nous spécifions que nous voulons charger <br />
uniquement les fichiers dont l'extension est **jpg**.

Nous indiquons également de charger tous les objets possibles contenus <br />
dans les sous-dossiers du dossier communiqué.

In [8]:
images = spark.read.format("binaryFile") \
  .option("pathGlobFilter", "*.jpg") \
  .option("recursiveFileLookup", "true") \
  .load(PATH_Data)

<u>Affichage des 5 premières images contenant</u> :
 - le path de l'image
 - la date et heure de sa dernière modification
 - sa longueur
 - son contenu encodé en valeur hexadécimal

<u>Je ne conserve que le **path** de l'image et j'ajoute <br />
    une colonne contenant les **labels** de chaque image</u> :

In [9]:
images = images.withColumn('label', element_at(split(images['path'], '/'),-2))
print(images.printSchema())
print(images.select('path','label').show(5,False))

root
 |-- path: string (nullable = true)
 |-- modificationTime: timestamp (nullable = true)
 |-- length: long (nullable = true)
 |-- content: binary (nullable = true)
 |-- label: string (nullable = true)

None


[Stage 0:>                                                          (0 + 1) / 1]

+----------------------------------------------------------------------------+-----+
|path                                                                        |label|
+----------------------------------------------------------------------------+-----+
|file:/home/steeve/Projets OCR/P9_BigData/Notebook/data/Test1/Kiwi/7_100.jpg |Kiwi |
|file:/home/steeve/Projets OCR/P9_BigData/Notebook/data/Test1/Kiwi/36_100.jpg|Kiwi |
|file:/home/steeve/Projets OCR/P9_BigData/Notebook/data/Test1/Kiwi/9_100.jpg |Kiwi |
|file:/home/steeve/Projets OCR/P9_BigData/Notebook/data/Test1/Kiwi/37_100.jpg|Kiwi |
|file:/home/steeve/Projets OCR/P9_BigData/Notebook/data/Test1/Kiwi/33_100.jpg|Kiwi |
+----------------------------------------------------------------------------+-----+
only showing top 5 rows
None


## 3.7. Préparation du modèle

Je vais utiliser la technique du **transfert learning** pour extraire les features des images.<br />
J'ai choisi d'utiliser le modèle **MobileNetV2** pour sa rapidité d'exécution comparée <br />
à d'autres modèles comme *VGG16* par exemple.

Il existe une dernière couche qui sert à classer les images <br />
selon 1000 catégories que nous ne voulons pas utiliser.<br />
L'idée dans ce projet est de récupérer le **vecteur de caractéristiques <br />
de dimensions (1,1,1280)** qui servira, plus tard, au travers d'un moteur <br />
de classification à reconnaitre les différents fruits du jeu de données.

Comme d'autres modèles similaires, **MobileNetV2**, lorsqu'on l'utilise <br />
en incluant toutes ses couches, attend obligatoirement des images <br />
de dimension (224,224,3). Nos images étant toutes de dimension (100,100,3), <br />
nous devrons simplement les **redimensionner** avant de les confier au modèle.

<u>Dans l'odre</u> :
 1. Nous chargeons le modèle **MobileNetV2** avec les poids **précalculés** <br />
    issus d'**imagenet** et en spécifiant le format de nos images en entrée
 2. Nous créons un nouveau modèle avec:
  - <u>en entrée</u> : l'entrée du modèle MobileNetV2
  - <u>en sortie</u> : l'avant dernière couche du modèle MobileNetV2

In [10]:
model = MobileNetV2(weights='imagenet',
                    include_top=True,
                    input_shape=(224, 224, 3))

E0000 00:00:1781263263.884302   98435 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [11]:
new_model = Model(inputs=model.input,
                  outputs=model.layers[-2].output)

Affichage du résumé de notre nouveau modèle où nous constatons <br />
que <u>nous récupérons bien en sortie un vecteur de dimension (1, 1, 1280)</u> :

In [12]:
new_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 2,223,872 (8.48 MB)

 Non-trainable params: 34,112 (133.25 KB)

Tous les workeurs doivent pouvoir accéder au modèle ainsi qu'à ses poids. <br />
Une bonne pratique consiste à charger le modèle sur le driver puis à diffuser <br />
ensuite les poids aux différents workeurs.

In [13]:
broadcast_weights = sc.broadcast(new_model.get_weights())

<u>Mettons cela sous forme de fonction</u> :

In [14]:
def model_fn():
    """
    Returns a MobileNetV2 model with top layer removed 
    and broadcasted pretrained weights.
    """
    model = MobileNetV2(weights='imagenet',
                        include_top=True,
                        input_shape=(224, 224, 3))
    for layer in model.layers:
        layer.trainable = False
    new_model = Model(inputs=model.input,
                  outputs=model.layers[-2].output)
    new_model.set_weights(broadcast_weights.value)
    return new_model

## 3.7. Définition du processus de chargement des images et application <br/>de leur featurisation à travers l'utilisation de pandas UDF

Ce notebook définit la logique par étapes, jusqu'à Pandas UDF.

<u>L'empilement des appels est la suivante</u> :

- Pandas UDF
  - featuriser une série d'images pd.Series
   - prétraiter une image

In [15]:
def preprocess(content):
    """
    Preprocesses raw image bytes for prediction.
    """
    img = Image.open(io.BytesIO(content)).resize([224, 224])
    arr = img_to_array(img)
    return preprocess_input(arr)

def featurize_series(model, content_series):
    """
    Featurize a pd.Series of raw images using the input model.
    :return: a pd.Series of image features
    """
    input = np.stack(content_series.map(preprocess))
    preds = model.predict(input)
    # For some layers, output features will be multi-dimensional tensors.
    # We flatten the feature tensors to vectors for easier storage in Spark DataFrames.
    output = [p.flatten() for p in preds]
    return pd.Series(output)

@pandas_udf('array<float>', PandasUDFType.SCALAR_ITER)
def featurize_udf(content_series_iter):
    '''
    This method is a Scalar Iterator pandas UDF wrapping our featurization function.
    The decorator specifies that this returns a Spark DataFrame column of type ArrayType(FloatType).

    :param content_series_iter: This argument is an iterator over batches of data, where each batch
                              is a pandas Series of image data.
    '''
    # With Scalar Iterator pandas UDFs, we can load the model once and then re-use it
    # for multiple data batches.  This amortizes the overhead of loading big models.
    model = model_fn()
    for content_series in content_series_iter:
        yield featurize_series(model, content_series)

/home/steeve/miniconda3/envs/P9bigdata/lib/python3.10/site-packages/pyspark/sql/pandas/functions.py:761: UserWarning: In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.
  warnings.warn(


## 3.7 Exécution des actions d'extraction de features

Les Pandas UDF, sur de grands enregistrements (par exemple, de très grandes images), <br />
peuvent rencontrer des erreurs de type Out Of Memory (OOM).<br />
Si vous rencontrez de telles erreurs dans la cellule ci-dessous, <br />
essayez de réduire la taille du lot Arrow via 'maxRecordsPerBatch'

Je n'utiliserai pas cette commande dans ce projet <br />
et je laisse donc la commande en commentaire.

In [16]:
# spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "1024")

Nous pouvons maintenant exécuter la featurisation sur l'ensemble de notre DataFrame Spark.<br />
<u>REMARQUE</u> : Cela peut prendre beaucoup de temps, tout dépend du volume de données à traiter. <br />

Notre jeu de données de **Test** contient **22819 images**. <br />
Cependant, dans l'exécution en mode **local**, <br />
nous <u>traiterons un ensemble réduit de **300 images**</u>.

In [17]:
features_df = images.repartition(20).select(col("path"),
                                            col("label"),
                                            featurize_udf("content").alias("features")
                                           )

<u>Rappel du PATH où seront inscrits les fichiers au format "**parquet**" <br />
contenant nos résultats, à savoir, un DataFrame contenant 3 colonnes</u> :
 1. Path des images
 2. Label de l'image
 3. Vecteur de caractéristiques de l'image

In [18]:
print(PATH_Result)

features_df.printSchema()

/home/steeve/Projets OCR/P9_BigData/Notebook/data/Results
root
 |-- path: string (nullable = true)
 |-- label: string (nullable = true)
 |-- features: array (nullable = true)
 |    |-- element: float (containsNull = true)



## 3.7 Réduction de dimension des features par PCA

La bibliothèque Spark ML exige un objet VectorUDT pour les algorithmes de Machine Learning.  
  
Une étape de conversion array<float> → VectorUDT est donc nécessaire <br />
avant d'appliquer la PCA.

<u>Créer une fonction Spark qui prend un tableau (arr) et le transforme en vecteur dense Spark ML</u>.

In [22]:
array_to_vector_udf = udf(lambda arr: Vectors.dense(arr), VectorUDT())

<u>Créer un nouveau DataFrame contenant une colonne supplémentaire appelée features_vector

In [21]:
features_vectorized_df = features_df.withColumn(
    "features_vector",
    array_to_vector_udf(col("features"))
)

In [23]:
features_vectorized_df.rdd.getNumPartitions()

[Stage 1:==============================================>           (8 + 1) / 10]

20

Vérification des types. La colonne "features_vector" est bien un vecteurUDF  
Nous utiliserons cette colonne pour la PCA. 

In [26]:
features_vectorized_df.dtypes

[('path', 'string'),
 ('label', 'string'),
 ('features', 'array<float>'),
 ('features_vector', 'vector')]

<u>Détermination du nombre de composantes PCA </u>:
   
Avec Spark ML, il faut obligatoirement fournir le nombre de composantes<br />principales `k` à conserver.   
Afin de choisir une valeur pertinente, une première PCA est entraînée avec<br />
un nombre élevé de composantes.  
  
La variance expliquée cumulée est calculée dans le but de mesurer la quantité<br />
d'information conservée lorsque le nombre de composantes augmente.  
  
Sa lecture permet alors d'identifier le nombre minimal de composantes<br />
conservant une part suffisante de l'information.  
  
Le nombre de composantes retenu correspond ainsi à un compromis entre :  
- la réduction de la dimension des données ;
- la conservation de l'information contenue dans les vecteurs de caractéristiques ;
- les performances de stockage et de calcul.

In [27]:
features_vectorized_df = features_vectorized_df.cache()
features_vectorized_df.count()

pca_var = PCA(k=200,
          inputCol="features_vector",
          outputCol="pca_features"
         )
model_pca_var = pca_var.fit(features_vectorized_df)

explained_var = model_pca_var.explainedVariance
cum_var = np.cumsum(explained_var)

I0000 00:00:1781263650.760058   99026 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781263651.201915   99026 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781263657.926036   99026 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1781263662.452341   99026 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━

50 0.9613606248750307
70 0.973165322198264
90 0.9807021491768713
110 0.9859659806385311
130 0.9897466243670863
150 0.9925420478104298
170 0.9946525881129552
190 0.996254451291243


In [30]:
for i in np.arange(10, 200, 10):
    print(i, cum_var[i-1])

10 0.8641197967324117
20 0.9163019778543455
30 0.9391439750786964
40 0.9523711356950714
50 0.9613606248750307
60 0.9680336074566754
70 0.973165322198264
80 0.9772953654238544
90 0.9807021491768713
100 0.9835695585304416
110 0.9859659806385311
120 0.9879917842100683
130 0.9897466243670863
140 0.9912560106134808
150 0.9925420478104298
160 0.993667452048952
170 0.9946525881129552
180 0.9955110026365195
190 0.996254451291243


In [34]:
seuil = 0.95
k_optimal = np.argmax(cum_var >= seuil) + 1
print(
    f"{seuil:.0%} de variance expliquée atteints avec "
    f"{k_optimal} composantes"
)

95% de variance expliquée atteints avec 38 composantes


Cela représente une réduction satisfaisante depuis 1280 dimensions <br />
tout en conservant la quasi-totalité de l'information.

In [35]:
pca = PCA(k=k_optimal,
          inputCol="features_vector",
          outputCol="pca_features"
         ) # Crée une PCA qui réduit les 1280 dimensions en k Composantes Principales

# Entraîne la PCA sur les vecteurs de la colonne features.
model_pca = pca.fit(features_vectorized_df)

# Appliquation la PCA au DataFrame
features_pca_df = model_pca.transform(features_vectorized_df)

# affiche le vecteur PCA de la première ligne.
# features_pca_df.select("pca_features").show(1, truncate=False)

# Donne la part de variance expliquée par chaque composante principale.
# model_pca.explainedVariance 


<u>En option, la sauvegarde de l'objet et du modèle pca </u>:  
(cellule en RAW)

<u>Enregistrement des données traitées au format "**parquet**"</u> :

In [41]:
features_pca_df.select("path", "label", "features", "pca_features") \
    .write.mode("overwrite") \
    .parquet(PATH_Result)

## 3.8 Chargement des données enregistrées et validation du résultat

<u>On charge les données fraichement enregistrées dans un **DataFrame Pandas**</u> :

In [44]:
df = pd.read_parquet(PATH_Result, engine='pyarrow')

<u>On affiche les 5 premières lignes du DataFrame</u> :

In [45]:
df.head()

,path,label,features,pca_features
0,file:/home/steeve/Projets OCR/P9_BigData/Noteb...,Kiwi,"[0.83250785, 0.0029626184, 0.0, 0.0, 0.0, 0.0,...","{'type': 1, 'size': None, 'indices': None, 'va..."
1,file:/home/steeve/Projets OCR/P9_BigData/Noteb...,Kiwi,"[0.47922668, 0.0, 0.0, 0.0035950504, 0.0, 0.00...","{'type': 1, 'size': None, 'indices': None, 'va..."
2,file:/home/steeve/Projets OCR/P9_BigData/Noteb...,Kiwi,"[0.70243555, 0.0, 0.0, 0.0, 0.00857977, 0.0, 0...","{'type': 1, 'size': None, 'indices': None, 'va..."
3,file:/home/steeve/Projets OCR/P9_BigData/Noteb...,Apricot,"[0.21265163, 0.18188705, 0.0, 0.0, 0.003698845...","{'type': 1, 'size': None, 'indices': None, 'va..."
4,file:/home/steeve/Projets OCR/P9_BigData/Noteb...,Apricot,"[1.0993154, 0.00024267119, 0.0, 0.0, 0.0863434...","{'type': 1, 'size': None, 'indices': None, 'va..."


La colonne "pca_features" contient des dictionnaires  
Nous allons récupérer uniquement la liste de vecteurs classés dans "values"  

In [46]:
df["pca_features"]=df["pca_features"].apply(lambda x: x["values"])
df.head()

,path,label,features,pca_features
0,file:/home/steeve/Projets OCR/P9_BigData/Noteb...,Kiwi,"[0.83250785, 0.0029626184, 0.0, 0.0, 0.0, 0.0,...","[-13.91755631546923, -3.825809819302543, 0.315..."
1,file:/home/steeve/Projets OCR/P9_BigData/Noteb...,Kiwi,"[0.47922668, 0.0, 0.0, 0.0035950504, 0.0, 0.00...","[-14.299597099744716, -4.313399875492536, 0.74..."
2,file:/home/steeve/Projets OCR/P9_BigData/Noteb...,Kiwi,"[0.70243555, 0.0, 0.0, 0.0, 0.00857977, 0.0, 0...","[-15.455335734930728, -4.7096581810414, 0.1320..."
3,file:/home/steeve/Projets OCR/P9_BigData/Noteb...,Apricot,"[0.21265163, 0.18188705, 0.0, 0.0, 0.003698845...","[-4.592885774794855, 14.274861282109901, -0.49..."
4,file:/home/steeve/Projets OCR/P9_BigData/Noteb...,Apricot,"[1.0993154, 0.00024267119, 0.0, 0.0, 0.0863434...","[-4.202481307926376, 13.49676289899572, -0.744..."


In [47]:
df.dtypes

path            object
label           object
features        object
pca_features    object
dtype: object

<u>On valide la dimension des vecteurs de caractéristiques des images </u> :

In [48]:
print("dimensions après pca : ",df.loc[0,"pca_features"].shape)
print("dimensions avant pca : ",df.loc[0,'features'].shape)

dimensions après pca :  (38,)
dimensions avant pca :  (1280,)


<u>Export de la réduction de dimension au format csv<u/>

In [51]:
df[["path", "label", "pca_features"]].to_csv(PATH_Result+'/pca_features.csv', index=False)

Nous venons de valider le processus sur un jeu de données allégé en local <br />
où nous avons simulé un cluster de machines en répartissant la charge de travail <br />
sur différents cœurs de processeur au sein d'une même machine.

Nous allons maintenant généraliser le processus en déployant notre solution <br />
sur un réel cluster de machines et nous travaillerons désormais sur la totalité <br />
des 22819 images de notre dossier "Test".